In [4]:
import os
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.tools import tool
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter




In [5]:
# Load the PDF document 
# #Trading Book-(156 day trading and swing trading the currency market technical and fundamental strategies to profit from market moves)

pdf_url = "https://cdn.oujdalibrary.com/books/156/156-day-trading-and-swing-trading-the-currency-market-technical-and-fundamental-strategies-to-profit-from-market-moves-(www.tawcer.com).pdf"
loader = PyPDFLoader(pdf_url)
documents = loader.load()


In [6]:
# Checking the number of loaded-Pages in the PDF
print(f"Total number of documents: {len(documents)}pages")


Total number of documents: 291pages


In [7]:
# Text splitting 
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len,
    add_start_index=True
    )
chunks = text_splitter.split_documents(documents)
print(f"Total number of split documents: {len(chunks)}chunks")


Total number of split documents: 705chunks


In [8]:
print(f"Total number of split documents: {len(chunks)} chunks from the original {len(documents)} pages")

Total number of split documents: 705 chunks from the original 291 pages


In [9]:
# for ollama embeddings
# first we need to install ollama and run the ollama server in the terminal using the command "ollama serve"
# then pull the embedding model using the command "ollama pull nomic-embed-text" in the terminal

# by doing this we can use the local ollama server to generate embeddings for our documents and store them in the Chroma vector store for semantic search and retrieval.

embedding = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434"
)

In [11]:
# To check the generated embedding for a query, we can use the embed_query method of the OllamaEmbeddings class.
#  This method takes a query string as input and returns the corresponding embedding vector.

embedding.embed_query("What is the best way to learn trading?")[:5]


[-0.015545909, 0.050891776, -0.19177379, -0.012247442, 0.06290805]

In [12]:
# Total length of the embedding vector for the query
# 768 dimensions for the nomic-embed-text model

len(embedding.embed_query("What is the best way to learn trading?"))

768

In [13]:
# Create a Chroma vector store
vector_store = Chroma(
    collection_name="trading_book",
    embedding_function=embedding,
    persist_directory="./chroma_db"
)
# Add the split documents to the vector store
vector_store.add_documents(chunks)


print("Documents have been added to the Chroma vector store and persisted to disk.")

Documents have been added to the Chroma vector store and persisted to disk.


In [14]:
# Query the vector store for relevant documents based on a user query

query = "What are some effective trading strategies for the currency market?"
retrieved_docs = vector_store.similarity_search(query, k=5)
print("Retrieved documents:")
for i, doc in enumerate(retrieved_docs):
    print(f"Document {i+1}: {doc.page_content[:200]}...")
    

Retrieved documents:
Document 1: Before implementing successful trading strategies, it is important to understand
what drives the movements of currencies in the foreign exchange market. The
best strategies tend to be the ones that co...
Document 2: most market moving economic data cross-market correlations, and unique currency
characteristics just to name a few. For technical traders, there are strategies for both
breakouts and range trading tha...
Document 3: 53
CHAPTER 4
A Deeper Look
at the FX Market
T
he next three chapters cover some of the unique studies that I have done on the
FX market that provide some telling details for both the novice and advanc...
Document 4: ix
PREFACE
D
ay Trading and Swing Trading the Currency Marketis one of the most popular books
for new and experienced forex traders. In the third edition, all of the content
has been updated with new ...
Document 5: can spend a very long period of time in a certain trading environment. Also, the
currency market obeys 

In [ ]:
import chromadb
from langchain_chroma import Chroma
# 1. Reconnect to your persistent directory

client = chromadb.PersistentClient(path="./chroma_db")

# 2. Get the collection by name
collection = client.get_collection(name="trading_book")

# 3. Check the total number of items
vector_count = collection.count()
print(f"Total vectors in collection: {vector_count}")

Total vectors in collection: 705


In [18]:
query = "What does the book say about Volume Profile or Delta spikes?"

# Run semantic search
docs = vector_store.similarity_search(query, k=3)

# Print results
for i, doc in enumerate(docs):
    print(f"\n--- Match {i+1} ---")
    print(doc.page_content[:300]) # Prints first 300 characters of the chunk


--- Match 1 ---
refers to media such as a CD or DVD that is not included in the version you purchased, you may download
this material at http://booksupport.wiley.com. For more information about Wiley products, visit
www.wiley.com.
Library of Congress Cataloging-in-Publication Data:
Names: Lien, Kathy, 1980- | Lien,

--- Match 2 ---
TRADE PARAMETERS FOR VARIOUS MARKET CONDITIONS
81
frame that traders should start looking at when their trading day starts are daily
charts, even if you are trading on a 5-minute time frame.
■ Step #1: Determine the Trading Environment
Rules to Determining Trading Environment
There are many differen

--- Match 3 ---
TECHNICAL TRADING STRATEGY: MULTIPLE TIME FRAME ANALYSIS
94
FIGURE 8.3 USDCAD Weekly Chart
Source: eSignal
overall downtrend. However, in order to increase the successfulness of this trade,
we want to make sure that CHFJPY is also in a downtrend on a daily basis. Taking
a look at Figure 8.6, we can 
